# Kaapi Bricks Demo — One-Click Setup

**Run this notebook on ANY Databricks workspace to set up the complete demo.**

What it does:
1. Creates catalog schema + volumes
2. Generates 14 Delta tables (200K orders, 37 stores, 28 products)
3. Generates 6 KA PDF documents
4. Creates po_line_items table for invoice demo
5. Prints instructions for KA, Genie, MAS creation
6. Generates app.yaml for deployment

**Time: ~10 minutes**

## Step 0: Configure for YOUR Workspace
**EDIT THESE TWO VALUES ONLY** — everything else is automatic.

In [ ]:
# ========================================
# EDIT THESE FOR YOUR WORKSPACE
# ========================================
CATALOG = "fe_india_gcc_catalog"         # Your Unity Catalog name
SQL_WAREHOUSE_ID = "13a6b534c35ffbcb"   # Your SQL Warehouse ID (SQL > Warehouses > copy ID)
# ========================================

SCHEMA = "kaapi_bricks"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
print(f"Catalog: {CATALOG}")
print(f"Schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {VOLUME_PATH}")
print(f"Warehouse: {SQL_WAREHOUSE_ID}")

In [ ]:
%pip install faker holidays numpy pandas pyarrow fpdf2 databricks-sdk mlflow[databricks]>=3.1
dbutils.library.restartPython()

In [ ]:
# Re-set after restart
CATALOG = "fe_india_gcc_catalog"         # SAME AS ABOVE
SQL_WAREHOUSE_ID = "13a6b534c35ffbcb"   # SAME AS ABOVE
SCHEMA = "kaapi_bricks"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

## Step 1: Create Schema + Volumes

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw_data")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.invoices")
print("Schema and volumes created.")

## Step 2: Generate Data (14 Delta Tables)
This takes ~3-5 minutes. Generates 200K orders, 15K customers, 37 stores, 28 products.

In [ ]:
# The generate_data.py script should be uploaded alongside this notebook.
# It uses the CATALOG and SCHEMA variables set above.
# If running from the same folder:
%run ./generate_data

## Step 3: Generate KA Documents (6 PDFs)

In [ ]:
%run ./generate_ka_documents

## Step 4: Create PO Line Items (for Invoice Demo)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.po_line_items (
  line_id STRING, po_id STRING, ingredient_id STRING, ingredient_name STRING,
  quantity DOUBLE, unit STRING, unit_price DOUBLE, line_total DOUBLE
)
""")

# Find a Coorg Coffee PO for Koramangala
pos = spark.sql(f"""
  SELECT p.po_id FROM {CATALOG}.{SCHEMA}.purchase_orders p
  JOIN {CATALOG}.{SCHEMA}.stores st ON p.store_id = st.store_id
  WHERE st.name = 'Kaapi Bricks Koramangala' AND p.supplier_id = 'SUP-001'
  ORDER BY p.order_date DESC LIMIT 1
""").collect()

if pos:
    po_id = pos[0]['po_id']
    spark.sql(f"""
      INSERT INTO {CATALOG}.{SCHEMA}.po_line_items VALUES
      ('pl-001', '{po_id}', 'ING-001', 'Coorg Arabica Beans', 25.0, 'kg', 1200.0, 30000.0),
      ('pl-002', '{po_id}', 'ING-004', 'Chicory', 15.0, 'kg', 400.0, 6000.0),
      ('pl-003', '{po_id}', 'ING-002', 'House Blend Pre-Mix', 25.0, 'kg', 820.0, 20500.0),
      ('pl-004', '{po_id}', 'ING-021', 'Paper Cups (200ml)', 10.0, 'case', 800.0, 8000.0),
      ('pl-005', '{po_id}', 'ING-022', 'Stirrer Sticks', 5.0, 'case', 300.0, 1500.0)
    """)
    print(f"Created PO line items for {po_id}")
else:
    print("WARNING: No PO found. Run Step 2 first.")

## Step 5: Create Lakebase Instance
Run this cell. If the instance already exists, it will skip.

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

try:
    inst = w.database.get_database_instance("kaapi-bricks")
    print(f"Lakebase 'kaapi-bricks' already exists. State: {inst.state}")
except Exception:
    print("Creating Lakebase instance 'kaapi-bricks'...")
    w.database.create_database_instance(name="kaapi-bricks", capacity="CU_1")
    print("Created. Wait 2-3 minutes for it to become AVAILABLE.")

## Step 6: Verify Data

In [ ]:
tables = ["stores", "products", "orders", "order_items", "customers", "ingredients",
          "suppliers", "promotions", "inventory_transactions", "purchase_orders",
          "toppings", "order_item_toppings", "promotion_redemptions", "po_line_items"]
print(f"{'Table':<30} {'Rows':>10}")
print("-" * 42)
for t in tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.{t}").collect()[0][0]
        print(f"{t:<30} {count:>10,}")
    except:
        print(f"{t:<30} {'MISSING':>10}")

# Check PDFs
import os
pdf_path = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/ka_documents"
try:
    pdfs = [f.name for f in dbutils.fs.ls(pdf_path) if f.name.endswith('.pdf')]
    print(f"\nKA Documents: {len(pdfs)} PDFs in {pdf_path}")
    for p in pdfs: print(f"  {p}")
except:
    print(f"\nWARNING: No PDFs found at {pdf_path}. Run Step 3.")

## Step 7: Create Agents (Manual — Do These in the UI)

Run this cell to see the instructions.

In [ ]:
host = w.config.host.rstrip('/')
print(f"""
╔══════════════════════════════════════════════════════════════╗
║              CREATE THESE 3 AGENTS IN THE UI               ║
╚══════════════════════════════════════════════════════════════╝

Open: {host}/#ml/agent-bricks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. KNOWLEDGE ASSISTANT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Name:        Kaapi Bricks Operations Knowledge
   Volume Path: /Volumes/{CATALOG}/{SCHEMA}/raw_data/ka_documents
   Description: Answers questions about Kaapi Bricks recipes,
                barista training, food safety, equipment maintenance,
                franchise operations, and supplier agreements.

   → Note the KA endpoint name: ka-XXXXXXXX-endpoint

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. GENIE SPACE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Name:      Kaapi Bricks Analytics
   Warehouse: (select your SQL warehouse)
   Tables:    {CATALOG}.{SCHEMA}.*
              (add ALL tables: stores, products, orders, order_items,
               customers, ingredients, suppliers, promotions,
               inventory_transactions, purchase_orders, toppings,
               order_item_toppings, promotion_redemptions, po_line_items)

   Sample Questions:
     - What are the top 5 selling drinks this month?
     - Show me revenue for the Koramangala store
     - Which ingredients are running low in inventory?

   → Note the Genie Space ID

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. SUPERVISOR AGENT (MAS)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Name:         Kaapi Bricks HQ
   Description:  Routes store queries to specialized agents
   Instructions: Route recipes/policies/training to knowledge_assistant.
                 Route sales/inventory/revenue to analytics.
                 Route planning/weather/staffing to operations_advisor.

   Agents:
     a) knowledge_assistant → KA from step 1
     b) analytics           → Genie from step 2
     c) operations_advisor  → MCP connection (add after Step 8)

   → Note the MAS endpoint name: mas-XXXXXXXX-endpoint
""")

## Step 8: Generate app.yaml

**After creating KA, Genie, and MAS above**, fill in the values below and run this cell. It generates the app.yaml you need for deployment.

In [ ]:
# ========================================
# FILL THESE IN FROM STEP 7
# ========================================
MAS_ENDPOINT = "mas-XXXXXXXX-endpoint"   # From MAS creation
KA_ENDPOINT = "ka-XXXXXXXX-endpoint"     # From KA creation
GENIE_SPACE_ID = "XXXXXXXX"              # From Genie creation
# ========================================

app_yaml = f"""command: ["python", "app.py"]

env:
  - name: MAS_ENDPOINT_NAME
    value: "{MAS_ENDPOINT}"
  - name: LAKEBASE_INSTANCE_NAME
    value: "kaapi-bricks"
  - name: LAKEBASE_DATABASE_NAME
    value: "databricks_postgres"
  - name: SQL_WAREHOUSE_ID
    value: "{SQL_WAREHOUSE_ID}"

resources:
  - name: mas-endpoint
    serving_endpoint:
      name: "{MAS_ENDPOINT}"
      permission: CAN_QUERY
  - name: ka-endpoint
    serving_endpoint:
      name: "{KA_ENDPOINT}"
      permission: CAN_QUERY
  - name: genie-space
    genie_space:
      space_id: "{GENIE_SPACE_ID}"
      permission: CAN_EDIT
  - name: embedding-endpoint
    serving_endpoint:
      name: "databricks-bge-large-en"
      permission: CAN_QUERY
  - name: lakebase
    database:
      instance_name: "kaapi-bricks"
      database_name: "databricks_postgres"
      permission: CAN_CONNECT_AND_CREATE
  - name: sql-warehouse
    sql_warehouse:
      id: "{SQL_WAREHOUSE_ID}"
      permission: CAN_USE
  - name: llama-endpoint
    serving_endpoint:
      name: "databricks-meta-llama-3-3-70b-instruct"
      permission: CAN_QUERY
"""

print(app_yaml)
print("\n" + "=" * 60)
print("Copy the above into apps/main-chat-app/app.yaml")
print("Then run from terminal:")
print(f"  databricks apps create {main_app_name} --profile YOUR_PROFILE" if 'main_app_name' in dir() else "  databricks apps create kaapi-bricks-finale --profile YOUR_PROFILE")
print(f"  databricks sync ./apps/main-chat-app /Workspace/Users/YOU/kaapi-bricks-finale --profile YOUR_PROFILE")
print(f"  databricks apps deploy kaapi-bricks-finale --source-code-path /Workspace/Users/YOU/kaapi-bricks-finale --profile YOUR_PROFILE")

## Step 9: MCP Connection (for Operations Advisor)

After deploying the MCP server app, create the UC HTTP Connection.

**Prerequisites:**
1. Deploy `apps/mcp-server/` as a Databricks App named `kaapi-ops-mcp`
2. Create a Service Principal with an OAuth secret (Account Console)
3. Grant the SP `CAN_MANAGE` on the `kaapi-ops-mcp` app

In [ ]:
# ========================================
# FILL THESE IN
# ========================================
MCP_APP_URL = "https://kaapi-ops-mcp-XXXXXXX.aws.databricksapps.com"  # Your MCP app URL
SP_CLIENT_ID = "YOUR-SP-CLIENT-ID"       # Service Principal client ID
SP_SECRET = "YOUR-SP-SECRET"             # Service Principal secret
TOKEN_ENDPOINT = f"{host}/oidc/v1/token"  # Auto-derived
# ========================================

print(f"""
Run this SQL in a notebook or SQL editor:

CREATE CONNECTION kaapi_ops_mcp TYPE HTTP OPTIONS (
  host '{MCP_APP_URL}',
  port '443',
  base_path '/mcp',
  client_id '{SP_CLIENT_ID}',
  client_secret '{SP_SECRET}',
  oauth_scope 'all-apis',
  token_endpoint '{TOKEN_ENDPOINT}',
  is_mcp_connection 'true'
);

GRANT ALL PRIVILEGES ON CONNECTION kaapi_ops_mcp TO `account users`;

Then add to MAS Supervisor:
  Agent name: operations_advisor
  Type: MCP Connection
  Connection: kaapi_ops_mcp
  Description: Handles operational planning, weather impact, holiday/festival planning,
               staffing, demand forecasting for upcoming days.
""")

## Step 10: Update app.py Catalog References

The main-chat-app `app.py` has hardcoded catalog references. If your catalog is NOT `fevm_cme_conde_catalog`, update these lines:

```
Line ~545: INSERT INTO fevm_cme_conde_catalog.kaapi_bricks.inference_logs
Line ~812: CATALOG_SCHEMA = "fevm_cme_conde_catalog.kaapi_bricks"
Line ~811: INVOICE_VOLUME = "/Volumes/fevm_cme_conde_catalog/kaapi_bricks/invoices"
```

Replace `fevm_cme_conde_catalog` with your catalog name everywhere.

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════╗
║                    SETUP COMPLETE                           ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  Catalog:    {CATALOG:<40}  ║
║  Schema:     {SCHEMA:<40}  ║
║  Warehouse:  {SQL_WAREHOUSE_ID:<40}  ║
║  Lakebase:   kaapi-bricks{' ':<29}  ║
║                                                            ║
║  REMAINING MANUAL STEPS:                                   ║
║  [ ] Create KA in Agent Bricks UI                          ║
║  [ ] Create Genie Space in SQL UI                          ║
║  [ ] Create MAS Supervisor in Agent Bricks UI              ║
║  [ ] Deploy main-chat-app (databricks apps deploy)         ║
║  [ ] Deploy mcp-server app (databricks apps deploy)        ║
║  [ ] Create UC HTTP Connection for MCP                     ║
║  [ ] Add MCP agent to MAS                                  ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")